# 例 7.5 · 估计 GLD 与 GDX 价差的半衰期

例 7.2 检验了两只 ETF 的价格是否协整。本例接着问：价差偏离均值后，模型预计需要多少个交易日才会消除一半偏离？[MATLAB 原例](../matlab/example7_5.m) · [Python 实现](../python/examples/example7_5.py)

In [ ]:
import sys
from pathlib import Path

import numpy as np

repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / 'python' / 'epchan').is_dir()
)
sys.path.insert(0, str(repo_root / 'python'))

from epchan.get_tday import get_tday
from epchan.ols import ols
from epchan.paths import data_path

## 1. 对齐 GLD 和 GDX 的复权收盘价

按日期升序排列共同交易日，只保留两只 ETF 的价格均为有限值的记录。

In [ ]:
gld_dates, gld_prices = get_tday(data_path('GLD.xls'))
gdx_dates, gdx_prices = get_tday(data_path('GDX.xls'))
dates, gld_idx, gdx_idx = np.intersect1d(gld_dates, gdx_dates, return_indices=True)
gld, gdx = gld_prices[gld_idx], gdx_prices[gdx_idx]
valid = np.isfinite(gld) & np.isfinite(gdx)
dates, gld, gdx = dates[valid], gld[valid], gdx[valid]
print(f'共同有效交易日共 {len(dates)} 天，从 {dates[0]} 到 {dates[-1]}。')

## 2. 估计对冲比率并计算价差

用无截距回归拟合 $GLD_t = \beta GDX_t + z_t$。每做多 1 股 GLD、做空 $\beta$ 股 GDX，按复权价格计算的价差就是回归残差 $z_t$。

In [ ]:
fit = ols(gld, gdx)
hedge_ratio = float(fit.beta[0, 0])
spread = fit.resid.ravel()
print(f'对冲比率 = {hedge_ratio:.4f} 股 GDX / 股 GLD')

## 3. 估计均值回复速度与半衰期

将相邻交易日的价差变化 $z_t-z_{t-1}$ 对上一日的价差偏离做无截距回归。偏离量等于 $z_{t-1}$ 减去滞后价差序列的样本均值。斜率 $\theta$ 为负时，模型预计价差向均值回归。按原例的连续时间近似，半衰期为 $-\ln(2)/\theta$，单位是交易日。

In [ ]:
previous_spread = spread[:-1]
change = np.diff(spread)
deviation = previous_spread - previous_spread.mean()
theta = float(ols(change, deviation).beta[0, 0])
half_life = -np.log(2) / theta
print(f'均值回复回归斜率 θ = {theta:.6f}')
print(f'价差半衰期 = {half_life:.4f} 个交易日')

### 结果解读

用仓库数据估计，对冲比率为 **1.6395**，价差偏离均值的幅度预计经过约 **7.84 个交易日**缩小一半。书中记录的对冲比率为 1.6766，半衰期为 10.0037 个交易日。